# ETF Data Collection

This notebook collects the ETF data required for the risk-scoring framework, including prices, number of holdings, sector exposure, country exposure and currency information.

In [2]:
from pathlib import Path

import pandas as pd
import yfinance as yf

In [3]:
test_etf = yf.Ticker("VWRP.L")

test_prices = test_etf.history(
    period="1mo",
    interval="1d",
    auto_adjust=True
)

test_prices.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,
2026-06-22 00:00:00+01:00,143.820007,144.259995,143.139999,143.339996,430645,0.0,0.0,0.0
2026-06-23 00:00:00+01:00,141.100006,142.000000,140.472885,141.399994,449323,0.0,0.0,0.0
2026-06-24 00:00:00+01:00,141.460007,142.339996,141.080002,142.059998,315961,0.0,0.0,0.0
2026-06-25 00:00:00+01:00,142.320007,143.279999,140.759995,141.479996,423099,0.0,0.0,0.0
2026-06-26 00:00:00+01:00,141.320007,141.320007,139.800003,141.139999,428472,0.0,0.0,0.0


In [4]:
raw_data_path = Path("../data/raw")
raw_data_path.mkdir(parents=True, exist_ok=True)

test_prices.to_csv(raw_data_path / "VWRP_test_prices.csv")

print("Test data saved successfully.")

Test data saved successfully.


## ETF Universe

The project will use a selected group of ETFs across equities, bonds and alternatives. This table stores the ETF names, tickers, providers and classifications used throughout the analysis.

In [5]:
etf_columns = [
    "ETF_ID",
    "ETF_Name",
    "Ticker",
    "Provider",
    "Asset_Class",
    "Region",
    "Trading_Currency",
    "Number_of_Holdings"
]

etf_universe = pd.DataFrame(columns=etf_columns)

etf_universe

,ETF_ID,ETF_Name,Ticker,Provider,Asset_Class,Region,Trading_Currency,Number_of_Holdings


In [6]:
etf_universe = pd.read_excel("../data/raw/ETF_Risk_Dataset.xlsx")

etf_universe.head()

,ETF_ID,ETF_Name,Ticker,Provider,Risk_Category,Asset_Class,Region,Sector_Focus,Number_of_Holdings,Holdings_Data_Quality,Main_Currency_Exposure,Currency_Risk_Level,Data_As_Of,Source_URL,Notes
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Vanguard,Medium,Equity,Global,Broad,3782.0,Exact provider,USD-dominant / multi-currency,Medium,2026-06-30,https://www.vanguard.co.uk/uk-fund-directory/p...,Broad global equity ETF. Exact country split i...
1,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,iShares,Medium,Equity,Developed markets,Broad,1283.0,Exact provider,USD-dominant / multi-currency developed markets,Medium,2026-07-17,https://www.ishares.com/uk/individual/en/produ...,Developed-market equity ETF. Use provider hold...
2,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Vanguard,Medium,Equity,US,Broad large-cap,504.0,Exact provider,USD,Medium,2026-06-30,https://www.vanguard.co.uk/professional/produc...,Tracks S&P 500; US equity and USD exposure.
3,ISF,iShares Core FTSE 100 UCITS ETF,ISF.L,iShares,Medium,Equity,UK,Broad large-cap,100.0,Exact provider,GBP-dominant,Low,2026-07-17,https://www.ishares.com/uk/individual/en/produ...,UK equity exposure; some constituent revenue m...
4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Invesco,Higher,Equity,US,Growth / Nasdaq,100.0,Benchmark-level,USD,Medium,2026-04-30,https://www.invesco.com/uk/en/financial-produc...,Nasdaq-100 exposure; high sector/growth concen...


In [7]:
print("Rows:", etf_universe.shape[0])
print("Columns:", etf_universe.shape[1])
print("\nColumn names:")
print(etf_universe.columns.tolist())

Rows: 20
Columns: 15

Column names:
['ETF_ID', 'ETF_Name', 'Ticker', 'Provider', 'Risk_Category', 'Asset_Class', 'Region', 'Sector_Focus', 'Number_of_Holdings', 'Holdings_Data_Quality', 'Main_Currency_Exposure', 'Currency_Risk_Level', 'Data_As_Of', 'Source_URL', 'Notes']


In [9]:
etf_universe.columns.tolist()

['ETF_ID',
 'ETF_Name',
 'Ticker',
 'Provider',
 'Risk_Category',
 'Asset_Class',
 'Region',
 'Sector_Focus',
 'Number_of_Holdings',
 'Holdings_Data_Quality',
 'Main_Currency_Exposure',
 'Currency_Risk_Level',
 'Data_As_Of',
 'Source_URL',
 'Notes']

In [10]:
display(
    etf_universe[
        [
            "ETF_ID",
            "ETF_Name",
            "Ticker",
            "Asset_Class",
            "Number_of_Holdings",
            "Main_Currency_Exposure"
        ]
    ]
)

print("\nMissing values:")
print(etf_universe.isna().sum())

,ETF_ID,ETF_Name,Ticker,Asset_Class,Number_of_Holdings,Main_Currency_Exposure
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,3782.0,USD-dominant / multi-currency
1,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,Equity,1283.0,USD-dominant / multi-currency developed markets
2,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Equity,504.0,USD
3,ISF,iShares Core FTSE 100 UCITS ETF,ISF.L,Equity,100.0,GBP-dominant
4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Equity,100.0,USD
5,VEUR,Vanguard FTSE Developed Europe UCITS ETF,VEUR.L,Equity,515.0,EUR/GBP/CHF/multi-Europe
6,VJPN,Vanguard FTSE Japan UCITS ETF,VJPN.L,Equity,476.0,JPY
7,VFEG,Vanguard FTSE Emerging Markets UCITS ETF,VFEG.L,Equity,2306.0,TWD/CNY/INR/BRL/ZAR/SAR/MXN/multi-EM
8,ICHN,iShares MSCI China UCITS ETF,ICHN.L,Equity,576.0,CNY/HKD/USD
9,IUIT,iShares S&P 500 Information Technology Sector ...,IUIT.L,Equity,74.0,USD



Missing values:
ETF_ID                    0
ETF_Name                  0
Ticker                    0
Provider                  0
Risk_Category             0
Asset_Class               0
Region                    0
Sector_Focus              0
Number_of_Holdings        1
Holdings_Data_Quality     0
Main_Currency_Exposure    0
Currency_Risk_Level       0
Data_As_Of                0
Source_URL                0
Notes                     0
dtype: int64


In [11]:
etf_universe["Ticker"] = etf_universe["Ticker"].astype("string").str.strip()

print("Missing tickers:", etf_universe["Ticker"].isna().sum())
print("Duplicate tickers:", etf_universe["Ticker"].duplicated().sum())

etf_universe[["ETF_ID", "ETF_Name", "Ticker"]]

Missing tickers: 0
Duplicate tickers: 0


,ETF_ID,ETF_Name,Ticker
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L
1,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L
2,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L
3,ISF,iShares Core FTSE 100 UCITS ETF,ISF.L
4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L
5,VEUR,Vanguard FTSE Developed Europe UCITS ETF,VEUR.L
6,VJPN,Vanguard FTSE Japan UCITS ETF,VJPN.L
7,VFEG,Vanguard FTSE Emerging Markets UCITS ETF,VFEG.L
8,ICHN,iShares MSCI China UCITS ETF,ICHN.L
9,IUIT,iShares S&P 500 Information Technology Sector ...,IUIT.L


In [12]:
price_series = []
failed_tickers = []

for ticker in etf_universe["Ticker"].dropna().unique():
    try:
        history = yf.Ticker(ticker).history(
            period="5y",
            interval="1d",
            auto_adjust=True
        )

        if history.empty:
            failed_tickers.append(ticker)
            continue

        price_series.append(history["Close"].rename(ticker))

    except Exception as error:
        print(f"Error downloading {ticker}: {error}")
        failed_tickers.append(ticker)

etf_prices = pd.concat(price_series, axis=1).sort_index()

print("Price table shape:", etf_prices.shape)
print("Failed tickers:", failed_tickers)

etf_prices.tail()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ICHN.L"}}}
$ICHN.L: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


Price table shape: (1262, 19)
Failed tickers: ['ICHN.L']


,VWRP.L,SWDA.L,VUAG.L,ISF.L,EQQQ.L,VEUR.L,VJPN.L,VFEG.L,IUIT.L,IUHC.L,INRG.L,VGOV.L,IGLO.L,IBTG.L,AGGG.L,CORP.L,GHYS.L,SGLN.L,IWDP.L
Date,,,,,,,,,,,,,,,,,,,
2026-07-16 00:00:00+01:00,140.419998,10715.0,108.360001,1030.599976,53012.0,42.264999,37.352501,64.209999,48.849998,13.045,797.75,15.487,86.709999,4.6650,4.2940,89.059998,88.660004,5776.0,1967.25
2026-07-17 00:00:00+01:00,139.039993,10622.0,107.320000,1033.000000,51934.0,42.209999,36.522499,63.020000,48.060001,13.125,783.50,15.511,86.930000,4.6675,4.2955,89.129997,88.400002,5783.0,1989.00
2026-07-20 00:00:00+01:00,139.619995,10649.0,107.699997,1025.400024,52537.0,42.040001,36.904999,63.790001,48.599998,12.925,782.75,15.398,86.559998,4.6625,4.2825,88.930000,88.220001,5801.0,1976.00
2026-07-21 00:00:00+01:00,140.860001,10726.0,108.300003,1032.199951,53210.0,42.410000,37.767502,64.760002,49.320000,12.875,798.00,15.420,86.349998,4.6605,4.2780,88.769997,88.260002,5913.0,1985.50
2026-07-22 00:00:00+01:00,141.080002,10749.0,108.500000,1043.400024,53244.0,42.665001,37.487499,64.709999,49.529999,12.930,803.50,15.410,86.254997,4.6565,4.2745,88.690002,88.120003,6031.0,1981.00


In [13]:
etf_universe.loc[
    etf_universe["Ticker"] == "ICHN.L",
    "Ticker"
] = "ICHN.AS"

china_test = yf.Ticker("ICHN.AS").history(
    period="5y",
    interval="1d",
    auto_adjust=True
)

print("Rows downloaded:", len(china_test))
china_test.tail()

Rows downloaded: 1281


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,
2026-07-16 00:00:00+02:00,5.606,5.612,5.568,5.582,248546,0.0,0.0,0.0
2026-07-17 00:00:00+02:00,5.421,5.447,5.400,5.436,98528,0.0,0.0,0.0
2026-07-20 00:00:00+02:00,5.568,5.596,5.556,5.596,357937,0.0,0.0,0.0
2026-07-21 00:00:00+02:00,5.627,5.635,5.561,5.569,234806,0.0,0.0,0.0
2026-07-22 00:00:00+02:00,5.509,5.532,5.504,5.519,176880,0.0,0.0,0.0


In [14]:
china_prices = china_test["Close"].rename("ICHN.AS")

# Remove exchange time zones so dates align correctly
etf_prices.index = etf_prices.index.tz_localize(None).normalize()
china_prices.index = china_prices.index.tz_localize(None).normalize()

etf_prices = pd.concat(
    [etf_prices, china_prices],
    axis=1
).sort_index()

print("Number of ETFs:", etf_prices.shape[1])
print("Price table shape:", etf_prices.shape)

Number of ETFs: 20
Price table shape: (1284, 20)


In [15]:
etf_prices.index.name = "Date"

etf_prices.to_csv(
    "../data/raw/etf_prices_5y.csv"
)

print("Saved successfully.")
print("Rows:", etf_prices.shape[0])
print("ETFs:", etf_prices.shape[1])

Saved successfully.
Rows: 1284
ETFs: 20
